<a href="https://colab.research.google.com/github/DmitriyKolesnikM8O/MOEX-Scripts/blob/main/MOEX%26EMA50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ===========================================
# Поиск акций MOEX возле EMA50 (ЧАСОВОЙ ТАЙМФРЕЙМ)
# Исправлено: живые цены, часовой пояс МСК, отсев незакрытого бара
# ===========================================

import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from tqdm import tqdm
import time

# -------------------------------------------
# Настройки
# -------------------------------------------

DISTANCE = 1.0          # Максимальное расстояние до EMA50 в %
INTERVAL = 60           # 60 минут (часовой график)
DELAY = 0.05            # Задержка между запросами

# Фиксируем часовой пояс МСК (UTC+3), чтобы избежать сдвигов на серверах/Colab
MSK_TZ = timezone(timedelta(hours=3))

# -------------------------------------------
# 1. Быстрый снимок рынка (Живые цены)
# -------------------------------------------
print("📊 Загружаем снимок рынка (живые цены прямо сейчас)...")

def get_market_snapshot():
    # Используем запрос из большого скрипта для получения цены LAST
    url = "https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities.json"
    params = {
        "iss.meta": "off",
        "iss.only": "securities,marketdata",
        "securities.columns": "SECID,SHORTNAME",
        "marketdata.columns": "SECID,LAST,VALTODAY",
    }
    r = requests.get(url, params=params, timeout=15)
    js = r.json()

    sec = pd.DataFrame(js["securities"]["data"], columns=js["securities"]["columns"])
    md = pd.DataFrame(js["marketdata"]["data"], columns=js["marketdata"]["columns"])

    df = sec.merge(md, on="SECID", how="left")
    df["LAST"] = pd.to_numeric(df["LAST"], errors="coerce")
    df["VALTODAY"] = pd.to_numeric(df["VALTODAY"], errors="coerce")

    # Оставляем только те акции, по которым сегодня есть торги и известна цена
    df = df[(df["VALTODAY"] > 0) & (df["LAST"].notna())].reset_index(drop=True)
    return df

stocks = get_market_snapshot()
print(f"✅ Найдено ликвидных акций с живыми ценами: {len(stocks)}")

# -------------------------------------------
# 2. Функция загрузки свечей
# -------------------------------------------
def fetch_candles(ticker, days_back=14, interval=60):
    till = datetime.now(MSK_TZ)
    since = till - timedelta(days=days_back)

    # Используем board TQBR для получения самых точных данных
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/{ticker}/candles.json"

    all_rows, start, columns = [], 0, None
    while True:
        params = {
            "from": since.strftime("%Y-%m-%d %H:%M:%S"),
            "till": till.strftime("%Y-%m-%d %H:%M:%S"),
            "interval": interval,
            "start": start,
            "iss.meta": "off"
        }
        r = requests.get(url, params=params, timeout=15)
        if r.status_code != 200:
            break
        js = r.json().get("candles", {})
        rows = js.get("data", [])
        if columns is None:
            columns = js.get("columns", [])
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < 500:
            break
        start += len(rows)

    if not all_rows or columns is None:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows, columns=columns)
    df["begin"] = pd.to_datetime(df["begin"])
    df["end"] = pd.to_datetime(df["end"])
    df = df.rename(columns={
        "open": "Open", "close": "Close", "high": "High",
        "low": "Low", "volume": "Volume", "begin": "Date", "end": "DateEnd"
    })
    df = df[["Date", "DateEnd", "Open", "High", "Low", "Close", "Volume"]].sort_values("Date")
    df = df.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)

    # ОТСЕВ НЕЗАКРЫТОГО БАРА (логика из большого скрипта)
    # Сравниваем время конца свечи с текущим МСК временем
    now_msk = pd.Timestamp.now(tz=MSK_TZ).tz_localize(None)
    if len(df) > 0 and df["DateEnd"].iloc[-1] > now_msk:
        df = df.iloc[:-1] # Убираем последнюю свечу, т.к. она еще в процессе

    return df.drop(columns=["DateEnd"]).reset_index(drop=True)

# -------------------------------------------
# 3. Основной цикл
# -------------------------------------------
results = []
print("\n🔍 Начинаем анализ акций (сравнение живой цены с EMA50)...\n")

for index, row in tqdm(stocks.iterrows(), total=len(stocks), desc="Обработка акций"):
    ticker = row["SECID"]
    name = row["SHORTNAME"]
    current_live_price = row["LAST"] # Берем цену ПРЯМО СЕЙЧАС из снипка

    try:
        df = fetch_candles(ticker, days_back=14, interval=INTERVAL)
        if df.empty or len(df) < 50:
            continue

        # Считаем EMA50 по закрытым свечам
        ema50 = df["Close"].ewm(span=50, adjust=False).mean().iloc[-1]

        # Считаем дистанцию между ЖИВОЙ ценой и актуальной EMA50
        distance = abs(current_live_price - ema50) / ema50 * 100

        if distance <= DISTANCE:
            results.append({
                "Тикер": ticker,
                "Компания": name,
                "Цена (LAST)": round(current_live_price, 4),
                "EMA50": round(ema50, 4),
                "Расстояние %": round(distance, 2),
                "Выше EMA": "✅" if current_live_price > ema50 else "❌",
            })

        time.sleep(DELAY)

    except Exception as e:
        continue

# -------------------------------------------
# 4. Вывод
# -------------------------------------------
print("\n" + "="*80)

if results:
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("Расстояние %")

    print(f"✅ НАЙДЕНО АКЦИЙ ВОЗЛЕ EMA50: {len(result_df)}")
    print("="*80)
    print("\n📊 Результаты:\n")
    print(result_df.to_string(index=False))

    filename = f"акции_возле_ema50_{datetime.now(MSK_TZ).strftime('%Y%m%d_%H%M')}.csv"
    result_df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\n💾 Сохранен в: {filename}")
else:
    print("❌ Акций возле EMA50 не найдено.")

print("="*80)

📊 Загружаем снимок рынка (живые цены прямо сейчас)...
✅ Найдено ликвидных акций с живыми ценами: 426

🔍 Начинаем анализ акций (сравнение живой цены с EMA50)...



Обработка акций: 100%|██████████| 426/426 [06:30<00:00,  1.09it/s]


✅ НАЙДЕНО АКЦИЙ ВОЗЛЕ EMA50: 268

📊 Результаты:

       Тикер   Компания  Цена (LAST)      EMA50  Расстояние % Выше EMA
        BAZA     iБАЗИС     113.2600   113.2463          0.01        ✅
RU000A10EVE8   ПАРУС-МВ     897.0000   896.9264          0.01        ✅
        KROT КрасОкт-ао    1002.0000  1002.0863          0.01        ❌
        FLOW   FLOW ETF    1016.6000  1016.4947          0.01        ✅
        MFGS  Мегион-ао     308.0000   307.9803          0.01        ✅
        AKUP   AKUP ETF      12.2200    12.2182          0.01        ✅
        SBLB   SBLB ETF      12.4560    12.4538          0.02        ✅
        FMBR   FMBR ETF      11.3020    11.2993          0.02        ✅
        KUZB КузнецкийБ       0.0290     0.0290          0.02        ❌
        SBGB   SBGB ETF      15.5150    15.5124          0.02        ✅
        AMFL   AMFL ETF     138.9000   138.8690          0.02        ✅
        AFLT   Аэрофлот      35.9200    35.9323          0.03        ❌
        PSMM   PSMM ETF    